## Step 1: Project Setup

In [1]:
"""
Titanic Data Analysis and JSON Export
Author: [Moritz Glas]
Description: Analyze Titanic passenger data, engineer features, and export to JSON
"""
 
import pandas as pd
import numpy as np
import json
from pathlib import Path
 
# Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_processed.json"
 
# Create data directory if it doesn't exist
DATA_DIR.mkdir(exist_ok=True)
 
print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")


Project setup complete!
Data directory: data
CSV file location: data/titanic.csv


## Step 2: Importing and Exploring the Data 

In [2]:
import pandas as pd
from pathlib import Path

# Load the CSV file into a pandas DataFrame
DATA_DIR = Path("data")
df = pd.read_csv(DATA_DIR / "titanic.csv")

In [3]:
print(f"Dataset loaded successfully! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

Dataset loaded successfully! Shape: (891, 12)

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

First few rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 175

## Step 3: Descriptive Statistics 

In [4]:
# Select numeric columns only (text columns cannot be averaged)
numeric_columns = df.select_dtypes(include="number")

# Calculate statistics (mean, median, std)
stats = numeric_columns.agg(["mean", "median", "std"])


In [5]:
# Display results
print(stats)

        PassengerId  Survived    Pclass        Age     SibSp     Parch  \
mean     446.000000  0.383838  2.308642  29.699118  0.523008  0.381594   
median   446.000000  0.000000  3.000000  28.000000  0.000000  0.000000   
std      257.353842  0.486592  0.836071  14.526497  1.102743  0.806057   

             Fare  
mean    32.204208  
median  14.454200  
std     49.693429  


In [6]:
# Display results
stats.T.round(2)

,mean,median,std
PassengerId,446.00,446.00,257.35
Survived,0.38,0.00,0.49
Pclass,2.31,3.00,0.84
Age,29.70,28.00,14.53
SibSp,0.52,0.00,1.10
Parch,0.38,0.00,0.81
Fare,32.20,14.45,49.69


## Step 4: Missing Values

In [7]:
# Count missing values
print("\n" + "="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)

missing_data = {}

for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_percent = (missing_count / len(df)) * 100

    # Only store columns that actually have missing values
    if missing_count > 0:
        missing_data[col] = {
            "missing_count": missing_count,
            "missing_percent": round(missing_percent, 2)
        }
        print(f"{col}: {missing_count} missing ({missing_percent:.2f}%)")




MISSING VALUES ANALYSIS
Age: 177 missing (19.87%)
Cabin: 687 missing (77.10%)
Embarked: 2 missing (0.22%)


## Step 5: Feature Engineering 

In [8]:
# Work on a copy so the original DataFrame stays unchanged
df_features = df.copy()

In [9]:
# FamilySize = siblings/spouses + parents/children + 1 (the passenger)
df_features['FamilySize'] = df_features['SibSp'] + df_features['Parch'] + 1 
print(df_features[['SibSp', 'Parch', 'FamilySize']].head(10))

   SibSp  Parch  FamilySize
0      1      0           2
1      1      0           2
2      0      0           1
3      1      0           2
4      0      0           1
5      0      0           1
6      0      0           1
7      3      1           5
8      0      2           3
9      1      0           2


In [10]:
# IsAlone = 1 if the passenger travelled without family, otherwise 0
df_features['IsAlone'] = (df_features['FamilySize'] == 1).astype(int)
print(df_features[['FamilySize', 'IsAlone']].head(10))

   FamilySize  IsAlone
0           2        0
1           2        0
2           1        1
3           2        0
4           1        1
5           1        1
6           1        1
7           5        0
8           3        0
9           2        0


In [11]:
df_features['IsAlone'] = (df_features['FamilySize'] == 1)
print(df_features[['FamilySize', 'IsAlone']].head(10))

   FamilySize  IsAlone
0           2    False
1           2    False
2           1     True
3           2    False
4           1     True
5           1     True
6           1     True
7           5    False
8           3    False
9           2    False


In [12]:
df_features['IsAlone'] = (df_features['FamilySize'] == 1).astype(int)
print(df_features[['FamilySize', 'IsAlone']].head(10))

   FamilySize  IsAlone
0           2        0
1           2        0
2           1        1
3           2        0
4           1        1
5           1        1
6           1        1
7           5        0
8           3        0
9           2        0


In [13]:
def categorize_age(age):
    """Categorize age into groups"""
    # Missing ages get their own group instead of being dropped
    if pd.isna(age):
        return 'Unknown'
    elif age < 18: 
        return 'Child'
    elif age < 30: 
        return 'Young Adult' 
    elif age < 50:
        return 'Adult'
    else: 
        return 'Senior' 
    

In [14]:
# Apply the function to every value in the Age column
df_features['AgeGroup'] = df_features['Age'].apply(categorize_age)
print(df_features[['Age', 'AgeGroup']].head(10))

    Age     AgeGroup
0  22.0  Young Adult
1  38.0        Adult
2  26.0  Young Adult
3  35.0        Adult
4  35.0        Adult
5   NaN      Unknown
6  54.0       Senior
7   2.0        Child
8  27.0  Young Adult
9  14.0        Child


In [15]:
# Compare family size between non-survivors (0) and survivors (1)
family_survival = df_features.groupby('Survived')['FamilySize'].agg(['mean', 'median', 'std'])
print(family_survival)

              mean  median       std
Survived                            
0         1.883424     1.0  1.830669
1         1.938596     2.0  1.186076


In [16]:
# Split the passengers into two groups using a boolean filter
survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]

In [17]:
print(f"  Difference: {abs(survived['FamilySize'].mean() - not_survived['FamilySize'].mean()):.2f}")

  Difference: 0.06


In [18]:
# Mean of a 0/1 column = share of survivors (survival rate)
print("\nSurvival rate by IsAlone:")
print(df_features.groupby('IsAlone')['Survived'].mean())

print("\nSurvival rate by AgeGroup:")
print(df_features.groupby('AgeGroup')['Survived'].mean())


Survival rate by IsAlone:
IsAlone
0    0.505650
1    0.303538
Name: Survived, dtype: float64

Survival rate by AgeGroup:
AgeGroup
Adult          0.417969
Child          0.539823
Senior         0.364865
Unknown        0.293785
Young Adult    0.350554
Name: Survived, dtype: float64


### Additional Feature: Title

In [19]:
# Extract the title between comma and dot, e.g. "Braund, Mr. Owen Harris" -> "Mr"
df_features['Title'] = df_features['Name'].str.split(', ').str[1].str.split('.').str[0]
print(df_features[['Name', 'Title']].head(10))

                                                Name   Title
0                            Braund, Mr. Owen Harris      Mr
1  Cumings, Mrs. John Bradley (Florence Briggs Th...     Mrs
2                             Heikkinen, Miss. Laina    Miss
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)     Mrs
4                           Allen, Mr. William Henry      Mr
5                                   Moran, Mr. James      Mr
6                            McCarthy, Mr. Timothy J      Mr
7                     Palsson, Master. Gosta Leonard  Master
8  Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)     Mrs
9                Nasser, Mrs. Nicholas (Adele Achem)     Mrs


In [20]:
print(df_features['Title'].value_counts())

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64


In [21]:
# Group rare titles (Dr, Rev, Countess, ...) as they are too few to analyse
def group_title(title):
    if title in ['Mr', 'Mrs', 'Miss', 'Master']:
        return title
    else:
        return 'Rare'

df_features['Title'] = df_features['Title'].apply(group_title)
print(df_features['Title'].value_counts())

Title
Mr        517
Miss      182
Mrs       125
Master     40
Rare       27
Name: count, dtype: int64


In [22]:
print("\nSurvival rate by Title:")
print(df_features.groupby('Title')['Survived'].mean())


Survival rate by Title:
Title
Master    0.575000
Miss      0.697802
Mr        0.156673
Mrs       0.792000
Rare      0.444444
Name: Survived, dtype: float64


This new feature which groups the passangers by title gives several insights at once. First of all it seperates women and men (Mr. vs. Miss and Mrs.) but also Men and boys (Mr. vs. Master). So the insights are much more detailed than the ones from FamilySize. 

In [23]:
# datetime is needed for the export date in the JSON metadata
import json
from datetime import datetime

## Step 6: Data Export Classes

In [24]:
class Passenger: 
    """Represents a single passenger with all their information"""
    def __init__(self, passenger_id, name, age, sex, survived, pclass, fare, embarked=None, family_size=None, is_alone=None, title=None):
        # Convert pandas types to plain Python types and NaN to None,
        # because JSON cannot store pandas types or NaN
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None 
        self.name = str(name) if pd.notna(name) else None 
        self.age = float(age) if pd.notna(age) else None 
        self.sex = str(sex) if pd.notna(sex) else None 
        self.survived = int(survived) if pd.notna(survived) else None 
        self.pclass = int(pclass) if pd.notna(pclass) else None 
        self.fare = float(fare) if pd.notna(fare) else None 
        self.embarked = str(embarked) if pd.notna(embarked) else None 
        self.family_size = int(family_size) if pd.notna(family_size) else None 
        self.is_alone = int(is_alone) if pd.notna(is_alone) else None 
        self.title = str(title) if pd.notna(title) else None 

    def to_dict(self): 
        """Convert passenger to dicitionary for JSON Serialization"""
        return {
            'passenger_id': self.passenger_id,
            'name': self.name, 
            'age': self.age,
            'sex': self.sex,
            'survived': self.survived,
            'pclass': self.pclass,
            'fare': self.fare,
            'embarked': self.embarked, 
            'family_size': self.family_size,
            'is_alone': self.is_alone,
            'title': self.title
        }
    



        

In [25]:
class TitanicDataset: 
    """Represents the entire Titanic dataset with methods for analysis and JSON export."""
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []
        self._create_passengers()
    def _create_passengers(self):
        """Create one Passenger object per row of the DataFrame."""
        for idx, row in self.dataframe.iterrows():
            passenger = Passenger(
                passenger_id=row.get('PassengerId', idx),
                name=row.get('Name'),
                age=row.get('Age'),
                sex=row.get('Sex'),
                survived=row.get('Survived'),
                pclass=row.get('Pclass'),
                fare=row.get('Fare'),
                embarked=row.get('Embarked'),
                family_size=row.get('FamilySize'),
                is_alone=row.get('IsAlone'),
                title=row.get('Title')
            )
            self.passengers.append(passenger)
    def get_missing_values(self):
        """Count missing values per column."""
        missing = {}
        for col in self.dataframe.columns:
            missing_count = int(self.dataframe[col].isnull().sum())
            if missing_count > 0:
                missing[col] = missing_count
        return missing 

    def get_summary_stats(self):
        """Calculate survival counts and average age and fare."""
        total = len(self.passengers) 
        survived_count = sum(1 for p in self.passengers if p.survived == 1)
        
        ages = [p.age for p in self.passengers if p.age is not None]
        fares = [p.fare for p in self.passengers if p.fare is not None]

        return {
            'total_passengers': total, 
            'survived': survived_count, 
            'did_not_survive': total - survived_count,
            'average_age': round(sum(ages) / len(ages), 2) if ages else None, 
            'average_fare': round(sum(fares) / len(fares), 2) if fares else None
        }
    def to_json(self, filename='titanic_data.json'):
        """Export metadata, statistics, missing values and all passengers to a JSON file."""
        data = {
            'metadata': {
                'dataset_name': 'Titanic Passenger Dataset',
                'export_date': datetime.now().isoformat(),
                'total_passengers': len(self.passengers),
                'survival_rate': round(float(self.dataframe['Survived'].mean()),4),
                'columns': list(self.dataframe.columns)
            },
            'summary_statistics': self.get_summary_stats(), 
            'missing_values': self.get_missing_values(),
            'passengers': [p.to_dict() for p in self.passengers]

        }
        with open(filename, 'w') as f:
            json.dump(data, f, indent=2)
        print(f"Data exported to {filename}")
        return data 
        

    


In [26]:
# The template expects the name df_engineered
df_engineered = df_features

if 'df_engineered' in locals() and not df_engineered.empty:
    # Creating the dataset automatically builds all Passenger objects
    dataset = TitanicDataset(df_engineered)

    print(f"Dataset created with {len(dataset.passengers)} passengers")
    print("\nFirst passenger:")
    print(dataset.passengers[0].to_dict())

    print("\nSummary statistics:")
    summary_stats = dataset.get_summary_stats()
    for key, value in summary_stats.items():
        print(f"  {key}: {value}")

    print("\nMissing values:")
    print(dataset.get_missing_values())

    dataset.to_json(JSON_FILE)

Dataset created with 891 passengers

First passenger:
{'passenger_id': 1, 'name': 'Braund, Mr. Owen Harris', 'age': 22.0, 'sex': 'male', 'survived': 0, 'pclass': 3, 'fare': 7.25, 'embarked': 'S', 'family_size': 2, 'is_alone': 0, 'title': 'Mr'}

Summary statistics:
  total_passengers: 891
  survived: 342
  did_not_survive: 549
  average_age: 29.7
  average_fare: 32.2

Missing values:
{'Age': 177, 'Cabin': 687, 'Embarked': 2}
Data exported to data/titanic_processed.json


## Step 7: Testing and Validation 

In [27]:
with open(JSON_FILE, 'r') as f:
    loaded = json.load(f)

print("JSON is valid!")
print(f"Passengers in file: {len(loaded['passengers'])}")
print(loaded['passengers'][0])

JSON is valid!
Passengers in file: 891
{'passenger_id': 1, 'name': 'Braund, Mr. Owen Harris', 'age': 22.0, 'sex': 'male', 'survived': 0, 'pclass': 3, 'fare': 7.25, 'embarked': 'S', 'family_size': 2, 'is_alone': 0, 'title': 'Mr'}


In [28]:
# Step 7: Load and inspect JSON
with open(JSON_FILE, 'r', encoding='utf-8') as f:
    json_data = json.load(f)

print("JSON loaded successfully!")

# 1. Are all main sections there?
expected_keys = ['metadata', 'summary_statistics', 'missing_values', 'passengers']
print("\nChecking main sections:")
for key in expected_keys:
    if key in json_data:
        print(f"  OK: {key}")
    else:
        print(f"  MISSING: {key}")

# 2. Is the number of passengers correct?
num_passengers = len(json_data['passengers'])
print(f"\nPassengers in JSON: {num_passengers}")
print(f"Passengers in DataFrame: {len(df_features)}")
if num_passengers == len(df_features):
    print("  OK: counts match")
else:
    print("  ERROR: counts do not match")

# 3. Does every passenger have all fields?
print("\nFields of first passenger:")
print(list(json_data['passengers'][0].keys()))

# 4. Statistics and missing values
print("\nMetadata:")
for key, value in json_data['metadata'].items():
    print(f"  {key}: {value}")

print("\nSummary statistics:")
for key, value in json_data['summary_statistics'].items():
    print(f"  {key}: {value}")

print("\nMissing values:")
for key, value in json_data['missing_values'].items():
    print(f"  {key}: {value}")

JSON loaded successfully!

Checking main sections:
  OK: metadata
  OK: summary_statistics
  OK: missing_values
  OK: passengers

Passengers in JSON: 891
Passengers in DataFrame: 891
  OK: counts match

Fields of first passenger:
['passenger_id', 'name', 'age', 'sex', 'survived', 'pclass', 'fare', 'embarked', 'family_size', 'is_alone', 'title']

Metadata:
  dataset_name: Titanic Passenger Dataset
  export_date: 2026-10-31T18:05:47.268404
  total_passengers: 891
  survival_rate: 0.3838
  columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'FamilySize', 'IsAlone', 'AgeGroup', 'Title']

Summary statistics:
  total_passengers: 891
  survived: 342
  did_not_survive: 549
  average_age: 29.7
  average_fare: 32.2

Missing values:
  Age: 177
  Cabin: 687
  Embarked: 2
